# Logistic Regression From Scratch — Phiên bản nâng cấp

**Thuật toán tối ưu:** Mini-batch Gradient Descent với Early Stopping & Learning Rate Decay

> **Các cải tiến:**
> - Random weight initialization nhỏ
> - Early Stopping
> - Learning Rate Decay
> - Threshold tối ưu trên Validation Set


### Import

In [37]:
import numpy as np
import pickle
from pathlib import Path
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
DATA_DIR = cwd / '..' / 'data'
if not DATA_DIR.exists():
    DATA_DIR = cwd / 'Labs' / 'Lab1' / 'data'
DATA_DIR = DATA_DIR.resolve()
print(f'DATA_DIR: {DATA_DIR}')


DATA_DIR: C:\Users\hary0\Documents\ML_Group6\ML_Group6\Labs\Lab1\data


### Load dữ liệu

In [38]:
X_all = load_npz(DATA_DIR / 'X_train.npz').toarray().astype(np.float32)
X_test = load_npz(DATA_DIR / 'X_test.npz').toarray().astype(np.float32)
y_all  = np.load(DATA_DIR / 'y_train.npy').astype(np.float32)
y_test = np.load(DATA_DIR / 'y_test.npy').astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=42
)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')


X_train: (3401, 3008) | X_val: (601, 3008) | X_test: (880, 3008)


### Implement LG_group6


In [39]:
class LG_group6:
    def __init__(self, learning_rate=0.1, n_epochs=100, batch_size=64,
                 lambda_reg=0.001, patience=10, lr_decay=0.95, random_state=42):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lambda_reg = lambda_reg
        self.patience = patience
        self.lr_decay = lr_decay
        self.random_state = random_state
        self.weights = None
        self.bias = 0.0
        self.loss_history = []
        self.val_loss_history = []

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _bce(self, y_true, y_pred):
        eps = 1e-9
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def fit(self, X, y, X_val=None, y_val=None):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.weights = np.random.randn(n_features).astype(np.float64) * 0.01
        self.bias = 0.0
        self.loss_history = []
        self.val_loss_history = []

        best_val_loss = np.inf
        patience_count = 0
        best_weights = self.weights.copy()
        best_bias = self.bias
        current_lr = self.lr

        for epoch in range(self.n_epochs):
            idx = np.random.permutation(n_samples)
            X_s, y_s = X[idx], y[idx]

            for start in range(0, n_samples, self.batch_size):
                Xb = X_s[start:start+self.batch_size]
                yb = y_s[start:start+self.batch_size]
                m = len(yb)

                z = Xb @ self.weights + self.bias
                y_hat = self._sigmoid(z)
                error = y_hat - yb

                grad_w = (Xb.T @ error) / m + self.lambda_reg * self.weights
                grad_b = np.mean(error)

                self.weights -= current_lr * grad_w
                self.bias -= current_lr * grad_b

            current_lr *= self.lr_decay

            y_pred_all = self._sigmoid(X @ self.weights + self.bias)
            train_loss = self._bce(y, y_pred_all)
            self.loss_history.append(train_loss)

            if X_val is not None:
                y_val_pred = self._sigmoid(X_val @ self.weights + self.bias)
                val_loss = self._bce(y_val, y_val_pred)
                self.val_loss_history.append(val_loss)

                if val_loss < best_val_loss - 1e-5:
                    best_val_loss = val_loss
                    best_weights = self.weights.copy()
                    best_bias = self.bias
                    patience_count = 0
                else:
                    patience_count += 1

                if patience_count >= self.patience:
                    self.weights = best_weights
                    self.bias = best_bias
                    break

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


### Huấn luyện Model

In [40]:
model = LG_group6(
    learning_rate=0.1,
    n_epochs=200,
    batch_size=64,
    lambda_reg=0.001,
    patience=15,
    lr_decay=0.97,
    random_state=42
)

model.fit(X_train, y_train, X_val=X_val, y_val=y_val)


### Các Hàm Đánh Giá

In [41]:
class DanhGia:
    @staticmethod
    def confusion_matrix(y_true, y_pred):
        TP = int(np.sum((y_pred==1)&(y_true==1)))
        FP = int(np.sum((y_pred==1)&(y_true==0)))
        TN = int(np.sum((y_pred==0)&(y_true==0)))
        FN = int(np.sum((y_pred==0)&(y_true==1)))
        return TP, FP, TN, FN

    @staticmethod
    def accuracy(y_true, y_pred): return np.mean(y_true == y_pred)

    @staticmethod
    def precision(TP, FP): return TP/(TP+FP) if (TP+FP)>0 else 0.0

    @staticmethod
    def recall(TP, FN): return TP/(TP+FN) if (TP+FN)>0 else 0.0

    @staticmethod
    def f1(precision, recall):
        return 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0

    @staticmethod
    def find_best_threshold(y_true, y_proba, metric='f1', thresholds=None):
        if thresholds is None:
            thresholds = np.linspace(0.05, 0.95, 181)
        best_val, best_t = 0.0, 0.5
        for t in thresholds:
            yp = (y_proba >= t).astype(int)
            TP, FP, TN, FN = DanhGia.confusion_matrix(y_true, yp)
            p = DanhGia.precision(TP, FP)
            r = DanhGia.recall(TP, FN)
            score = DanhGia.f1(p, r) if metric=='f1' else r
            if score > best_val:
                best_val, best_t = score, t
        return round(best_t, 3), round(best_val, 4)


### Tìm Threshold Tối Ưu trên Validation Set


In [42]:
y_val_proba = model.predict_proba(X_val)
BEST_THRESHOLD, best_f1_val = DanhGia.find_best_threshold(y_val, y_val_proba, metric='f1')
print(f'Threshold tối ưu: {BEST_THRESHOLD} | Val F1: {best_f1_val:.4f}')


Threshold tối ưu: 0.33 | Val F1: 0.9587


### Đánh Giá trên Test Set

In [43]:
y_proba = model.predict_proba(X_test)
y_pred = (y_proba >= BEST_THRESHOLD).astype(int)

TP, FP, TN, FN = DanhGia.confusion_matrix(y_test, y_pred)
acc = DanhGia.accuracy(y_test, y_pred)
prec = DanhGia.precision(TP, FP)
rec = DanhGia.recall(TP, FN)
f1 = DanhGia.f1(prec, rec)

print('KẾT QUẢ ĐÁNH GIÁ:')
print(f'Accuracy: {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'Confusion Matrix: TP={TP}, FP={FP}, TN={TN}, FN={FN}')


KẾT QUẢ ĐÁNH GIÁ:
Accuracy: 0.9511
Precision: 0.9309
Recall: 0.9579
F1-Score: 0.9442
Confusion Matrix: TP=364, FP=27, TN=473, FN=16


### Loss Curve (Train + Validation)

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(model.loss_history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, epochs_ran + 1), model.loss_history, color='#7F77DD', linewidth=2, label='Train Loss')
if model.val_loss_history:
    ax.plot(range(1, epochs_ran + 1), model.val_loss_history, color='#DD8452', linewidth=2, linestyle='--', label='Validation Loss')
ax.fill_between(range(1, epochs_ran + 1), model.loss_history, alpha=0.1, color='#7F77DD')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('Training & Validation Loss', fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Visualize — Confusion Matrix + Threshold Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = np.array([[TN, FP], [FN, TP]])
im = axes[0].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0].set_title('Confusion Matrix — LG_group6', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred Ham', 'Pred Spam'])
axes[0].set_yticklabels(['Actual Ham', 'Actual Spam'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f'{cm[i,j]}', ha='center', va='center', fontsize=14,
                     color='white' if cm[i,j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

ths = np.linspace(0.05, 0.95, 181)
accs, precs, recs, f1s = [], [], [], []
for t in ths:
    yp = (y_val_proba >= t).astype(int)
    tp, fp, tn, fn = DanhGia.confusion_matrix(y_val, yp)
    p = DanhGia.precision(tp, fp)
    r = DanhGia.recall(tp, fn)
    accs.append(DanhGia.accuracy(y_val, yp))
    precs.append(p)
    recs.append(r)
    f1s.append(DanhGia.f1(p, r))

axes[1].plot(ths, precs, label='Precision', color='#4C72B0')
axes[1].plot(ths, recs, label='Recall', color='#DD8452')
axes[1].plot(ths, f1s, label='F1', color='#55A868', linewidth=2.5)
axes[1].axvline(BEST_THRESHOLD, color='red', linestyle='--', label=f'Best={BEST_THRESHOLD}')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Threshold Analysis (Val Set)', fontweight='bold')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### So Sánh — LG_group6 vs Sklearn

In [ ]:
import time
from sklearn.linear_model import LogisticRegression

# Dùng class_weight='balanced' để so sánh công bằng hơn

t0 = time.time()
sklearn_model = LogisticRegression(max_iter=1000, random_state=42,
                                   solver='lbfgs', class_weight='balanced')
sklearn_model.fit(X_train, y_train)
sklearn_time = time.time() - t0
print(f'Sklearn train xong! Thời gian: {sklearn_time:.3f}s')

sk_proba = sklearn_model.predict_proba(X_test)[:, 1]
sk_val_proba = sklearn_model.predict_proba(X_val)[:, 1]
sk_best_thresh, _ = DanhGia.find_best_threshold(y_val, sk_val_proba)
sk_pred = (sk_proba >= sk_best_thresh).astype(int)

sk_TP, sk_FP, sk_TN, sk_FN = DanhGia.confusion_matrix(y_test, sk_pred)
sk_acc = DanhGia.accuracy(y_test, sk_pred)
sk_prec = DanhGia.precision(sk_TP, sk_FP)
sk_rec = DanhGia.recall(sk_TP, sk_FN)
sk_f1 = DanhGia.f1(sk_prec, sk_rec)

print('=' * 68)
print(f'  {"Metric":<12} | {"LG_group6":>14} | {"Sklearn":>14} | {"Diff":>10}')
print('=' * 68)
for name, s_val, sk_val in [('Accuracy', acc, sk_acc), ('Precision', prec, sk_prec),
                              ('Recall', rec, sk_rec), ('F1-Score', f1, sk_f1)]:
    diff = s_val - sk_val
    arrow = '▲' if diff > 0 else ('▼' if diff < 0 else '=')
    print(f'  {name:<12} | {s_val:>14.4f} | {sk_val:>14.4f} | {arrow} {abs(diff):.4f}')
print('=' * 68)

### ROC Curve + Precision-Recall Curve

In [ ]:
def compute_roc(y_true, y_scores, n=300):
    ths = np.linspace(0, 1, n)
    tprs, fprs = [], []
    for t in ths:
        yp = (y_scores >= t).astype(int)
        tp = int(np.sum((yp == 1) & (y_true == 1)))
        fp = int(np.sum((yp == 1) & (y_true == 0)))
        tn = int(np.sum((yp == 0) & (y_true == 0)))
        fn = int(np.sum((yp == 0) & (y_true == 1)))
        tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    return np.array(fprs), np.array(tprs)


def compute_pr(y_true, y_scores, n=300):
    ths = np.linspace(0, 1, n)
    ps, rs = [], []
    for t in ths:
        yp = (y_scores >= t).astype(int)
        tp = int(np.sum((yp == 1) & (y_true == 1)))
        fp = int(np.sum((yp == 1) & (y_true == 0)))
        fn = int(np.sum((yp == 0) & (y_true == 1)))
        ps.append(tp / (tp + fp) if (tp + fp) > 0 else 1.0)
        rs.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    return np.array(rs), np.array(ps)


def auc_trap(x, y):
    order = np.argsort(x)
    return float(np.trapezoid(y[order], x[order]))

fpr_s, tpr_s = compute_roc(y_test, y_proba)
fpr_sk, tpr_sk = compute_roc(y_test, sk_proba)
rec_s, pre_s = compute_pr(y_test, y_proba)
rec_sk, pre_sk = compute_pr(y_test, sk_proba)
auc_s = auc_trap(fpr_s, tpr_s)
auc_sk = auc_trap(fpr_sk, tpr_sk)
auc_pr_s = auc_trap(rec_s, pre_s)
auc_pr_sk = auc_trap(rec_sk, pre_sk)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].plot(fpr_s, tpr_s, color='#7F77DD', lw=2.5, label=f'LG_group6 (AUC={auc_s:.4f})')
axes[0].plot(fpr_sk, tpr_sk, color='#1D9E75', lw=2.5, ls='--', label=f'Sklearn (AUC={auc_sk:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', ls=':', lw=1, label='Random')
axes[0].fill_between(fpr_s, tpr_s, alpha=0.08, color='#7F77DD')
axes[0].fill_between(fpr_sk, tpr_sk, alpha=0.08, color='#1D9E75')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(ls='--', alpha=0.4)

axes[1].plot(rec_s, pre_s, color='#7F77DD', lw=2.5, label=f'LG_group6 (AP={auc_pr_s:.4f})')
axes[1].plot(rec_sk, pre_sk, color='#1D9E75', lw=2.5, ls='--', label=f'Sklearn (AP={auc_pr_sk:.4f})')
axes[1].fill_between(rec_s, pre_s, alpha=0.08, color='#7F77DD')
axes[1].fill_between(rec_sk, pre_sk, alpha=0.08, color='#1D9E75')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()
axes[1].grid(ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

print(f'ROC AUC  — LG_group6: {auc_s:.4f} | Sklearn: {auc_sk:.4f}')
print(f'PR  AUC  — LG_group6: {auc_pr_s:.4f} | Sklearn: {auc_pr_sk:.4f}')

### Experiment with Higher Thresholds (0.55–0.65)

In [ ]:
print('Thử nghiệm với threshold cao hơn để cải thiện Precision:')
print('=' * 70)
print(f'{"Threshold":<10} | {"Accuracy":>10} | {"Precision":>10} | {"Recall":>10} | {"F1":>10}')
print('=' * 70)
for thresh in [0.55, 0.60, 0.65]:
    y_pred_exp = (y_proba >= thresh).astype(int)
    TP_exp, FP_exp, TN_exp, FN_exp = DanhGia.confusion_matrix(y_test, y_pred_exp)
    acc_exp = DanhGia.accuracy(y_test, y_pred_exp)
    prec_exp = DanhGia.precision(TP_exp, FP_exp)
    rec_exp = DanhGia.recall(TP_exp, FN_exp)
    f1_exp = DanhGia.f1(prec_exp, rec_exp)
    print(f'{thresh:<10.2f} | {acc_exp:>10.4f} | {prec_exp:>10.4f} | {rec_exp:>10.4f} | {f1_exp:>10.4f}')
print('=' * 70)